# Семинар 08. Декораторы


## Цели

После семинара вы сможете:

- передавать функции как значения и объяснять работу замыканий;
- писать обычные и параметризованные декораторы;
- сохранять метаданные и поведение декорируемой функции;
- определять порядок применения нескольких декораторов;
- понимать ограничения логирования, повторных вызовов и декорирования `async`-функций.

## Перед началом

Понадобятся вложенные функции, области видимости, `*args`, `**kwargs` и исключения.


## Функции как объекты

В Python функция — обычный объект: её можно присвоить переменной, передать как аргумент, вернуть из другой функции и положить в коллекцию. Скобки меняют смысл: `func` — сам объект-функция, `func()` — его вызов.

Функция, которая принимает или возвращает другую функцию, называется функцией высшего порядка. Декораторы строятся именно на этом механизме.


In [ ]:
def add(left: int, right: int) -> int:
    return left + right


operation = add      # Функцию не вызываем.
assert operation(2, 3) == 5
assert operation is add

In [ ]:
from collections.abc import Callable


def make_counter() -> Callable[[], int]:
    count = 0

    def counter() -> int:
        nonlocal count
        count += 1
        return count

    return counter


first_counter = make_counter()
second_counter = make_counter()
assert first_counter() == 1
assert first_counter() == 2
assert second_counter() == 1

## Замыкание

Вложенная функция `counter` использует переменную `count` из внешней функции. После завершения `make_counter()` эта переменная не исчезает: возвращённая функция сохраняет к ней доступ. Такой набор функции и захваченного окружения называется замыканием. Каждый вызов `make_counter()` создаёт независимое состояние.

`nonlocal` нужен для присваивания переменной из ближайшей внешней области. Без него строка `count += 1` попыталась бы изменить ещё не созданную локальную переменную и закончилась бы `UnboundLocalError`. Замыкание хранит ссылки на захваченные объекты, поэтому крупный объект может оставаться в памяти дольше ожидаемого.

Замыкание захватывает переменную, а не снимок её значения. Поэтому функции, созданные в цикле, без дополнительной фиксации обычно увидят последнее значение переменной:

```python
callbacks = [lambda value=value: value for value in range(3)]
assert [callback() for callback in callbacks] == [0, 1, 2]
```

Параметр по умолчанию `value=value` вычисляется при создании каждой функции и фиксирует нужное значение. Без него все три функции вернули бы `2`.

## Синтаксис декоратора

Запись
```python
@decorator
def func():
    ...
```
примерно равна следующей:
```python
def func():
    ...

func = decorator(func)
```

Декоратор применяется при выполнении определения функции — обычно при импорте модуля, а не при первом вызове `func()`. Код внутри `wrapper` выполняется уже при каждом вызове.


In [ ]:
from typing import Any, Callable


def announce_call(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)

    return wrapper


@announce_call
def multiply(left: int, right: int) -> int:
    """Умножить два числа."""
    return left * right


assert multiply(3, 4) == 12
assert multiply.__name__ == "wrapper"  # Метаданные потеряны.

Декоратор — вызываемый объект, который получает декорируемый объект и возвращает объект, связанный с тем же именем. Обычно декоратор возвращает обёртку, но может зарегистрировать исходную функцию и вернуть её без изменений.

Декораторы используют для сквозной логики:

- логирования и сбора метрик;
- авторизации и проверки разрешений;
- повторения операций после ожидаемых временных ошибок;
- кэширования;
- регистрации обработчиков во фреймворках;
- реализации `@classmethod`, `@staticmethod`, `@property` и других встроенных механизмов.

Декоратор не должен незаметно ломать контракт функции. Если обёртка меняет аргументы, результат, исключения или синхронность вызова, это уже часть публичного API и её надо явно документировать.


In [ ]:
from functools import wraps
from typing import Any, Callable, TypeVar

R = TypeVar("R")


def trace(func: Callable[..., R]) -> Callable[..., R]:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> R:
        print(f"Before {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"After {func.__name__}")

    return wrapper


@trace
def divide(left: int, right: int) -> float:
    """Разделить left на right."""
    return left / right


assert divide(10, 2) == 5.0
assert divide.__name__ == "divide"
assert divide.__doc__ == "Разделить left на right."
assert divide.__wrapped__(10, 2) == 5.0

`functools.wraps` копирует имя, документацию, аннотации и другие метаданные, а также добавляет ссылку `__wrapped__` на исходную функцию. Без `wraps` отладчик, документация и инструменты интроспекции видят безликий `wrapper`. Это не косметика, а обязательная часть нормального декоратора.

`wraps` не исправляет сломанное поведение. Обёртка всё равно обязана корректно передать аргументы, вернуть результат и не подавить исключение без явного контракта. Для точного сохранения типов параметров статическая типизация использует `ParamSpec`; `Callable[..., R]` сохраняет тип результата, но не описывает исходную сигнатуру полностью.

## Параметризованный декоратор

Выражение `@retry(attempts=3)` сначала вызывает фабрику `retry(attempts=3)`. Она возвращает декоратор; декоратор получает функцию и возвращает обёртку. Здесь три уровня функций не прихоть, а прямое следствие трёх разных моментов вызова.


In [ ]:
from functools import wraps
from typing import Any, Callable, TypeVar

R = TypeVar("R")


def retry(
    attempts: int,
    exceptions: tuple[type[Exception], ...] = (TimeoutError,),
) -> Callable[[Callable[..., R]], Callable[..., R]]:
    if attempts < 1:
        raise ValueError("attempts must be positive")

    def decorator(func: Callable[..., R]) -> Callable[..., R]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> R:
            for attempt in range(1, attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as error:
                    if attempt == attempts:
                        raise
                    print(f"Attempt {attempt} failed: {error}")

            raise AssertionError("unreachable")

        return wrapper

    return decorator


outcomes = iter([TimeoutError("first"), TimeoutError("second"), "done"])


@retry(attempts=3, exceptions=(TimeoutError,))
def unstable_operation() -> str:
    outcome = next(outcomes)
    if isinstance(outcome, Exception):
        raise outcome
    return outcome


assert unstable_operation() == "done"

Перехватывать в `retry` любой `Exception` нельзя: `TypeError`, `AttributeError` и другие ошибки программы повторным вызовом не лечатся. Повторять следует только явно перечисленные временные ошибки. Операция также должна допускать повтор: слепой ретрай платежа или другой команды с побочным эффектом способен выполнить действие несколько раз.

Логирование всех `args` и `kwargs` тоже небезопасно по умолчанию: там могут оказаться пароли, токены и персональные данные. Такие поля надо маскировать или исключать.

Обычная синхронная обёртка не перехватывает исключения, возникающие во время `await` декорированной `async`-функции. Для неё нужен `async def wrapper(...)` с `return await func(...)`.

## Несколько декораторов

```python
@outer
@inner
def func():
    ...
```

эквивалентно `func = outer(inner(func))`: ближайший к функции `inner` применяется первым, а при вызове управление сначала попадает во внешний `outer`. Порядок критичен. Например, проверка прав доступа и кэширование в разном порядке дают разное поведение.

## Самопроверка

1. Чем объект функции отличается от результата её вызова?
2. Какое состояние сохраняет замыкание `make_counter()`?
3. Когда выполняется `decorator(func)`, а когда — тело `wrapper`?
4. Что именно сохраняет `functools.wraps` и чего он не гарантирует?
5. Почему нельзя без разбора повторять функцию после любого `Exception`?
6. В каком порядке применяются `@outer` и `@inner`?

## Итоги

- Функции — объекты; замыкания сохраняют доступ к переменным внешней области.
- Декоратор заменяет имя функции результатом своего вызова уже при определении функции.
- Нормальная обёртка передаёт аргументы, возвращает результат, по умолчанию не подменяет исключения и использует `functools.wraps`.
- Параметризованный декоратор состоит из фабрики, декоратора и обёртки.
- Ретраи, логирование, порядок декораторов и работа с `async` требуют явного контракта; магии здесь нет.

Документация Python: [определения функций и порядок применения декораторов](https://docs.python.org/3/reference/compound_stmts.html#function-definitions), [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps).


## Задание 1. Логирующий декоратор

Напишите декоратор, который при каждом вызове выводит имя функции, `args`, `kwargs` и результат. Если функция выбросила исключение, залогируйте его и пробросьте дальше.

```python
def log_call(func):
    ...
```

**Критерии проверки:** поддерживаются произвольные аргументы; исключения не подавляются; имя и документация исходной функции сохранены через `functools.wraps`.


## Задание 2. Параметризованный volkswagen-test декоратор

```python
def test_decorator(test_mode: bool = False):
    # test_mode=True: перехватить исключение и вернуть None
    # test_mode=False: полностью сохранить поведение исходной функции
    ...
```

**Критерии проверки:** обе ветви поведения покрыты тестами; успешный результат функции не меняется; метаданные исходной функции сохранены.
